In [ ]:
import mesa
import pandas as pd
import random
from datetime import datetime, timedelta

# --- EVENT LOGGER (Aligned with Bemthuis et al. Table 1 & Table 2) ---
class ProcessLogger:
    def __init__(self):
        self.log = []
        self.step_counter = 0 # To track event order within a single step

    def add_event(self, step, agent_id, activity, timestamp, **kwargs):
        self.step_counter += 1
        event = {
            "Step": step,                           # Matches paper's logical sequence
            "StepCounter": self.step_counter,       # Matches paper's ordering counter
            "CaseID": agent_id,                     # The Agent is the Case (as per the paper)
            "Activity": activity,                   # Uses concatenated naming convention
            "Timestamp": timestamp
        }
        event.update(kwargs)
        self.log.append(event)

    def to_dataframe(self):
        return pd.DataFrame(self.log)

# --- MINER AGENT ---
class MinerAgent(mesa.Agent):
    def __init__(self, model, pool_name, hash_power, energy_cost, strategy):
        super().__init__(model) 
        self.pool_name = pool_name         
        self.hash_power = hash_power       
        self.energy_cost = energy_cost     
        self.strategy = strategy           
        self.is_mining = True
        self.state = "IDLE" 
        self.reboot_timer = 0
        self.btc_balance = 0.0

    def step(self):
        # CHANGED: We now pull the step count directly from our custom model counter
        current_step = self.model.steps 
        current_time = self.model.current_time
        profitability = self.model.btc_price - self.energy_cost
        
        # Determine Profitability State for Concatenated Naming (e.g., 'changeHappy X_Y' style)
        profit_status = "HighProfit" if profitability > 500 else ("LowProfit" if profitability > 0 else "NegativeProfit")

        # 1. Behavioral Decision (Opportunist Strategy)
        if self.strategy == 'opportunist':
            if profitability < 0 and self.is_mining:
                self.is_mining = False
                self.state = "IDLE"
                # Concatenated Activity Naming
                self.model.logger.add_event(current_step, self.pool_name, f"Suspend_Hashing_{profit_status}", current_time, Strategy=self.strategy)
                return 
            elif profitability > 0 and not self.is_mining:
                self.is_mining = True
                self.model.logger.add_event(current_step, self.pool_name, f"Resume_Hashing_{profit_status}", current_time, Strategy=self.strategy)

        if not self.is_mining:
            return

        # 2. Outlier Behavior: Hardware Failures (Chaos injection)
        if random.random() < 0.005: # 0.5% chance of crash
            self.state = "REBOOTING"
            self.reboot_timer = random.randint(2, 5) 
            self.model.logger.add_event(current_step, self.pool_name, f"Hardware_Crash_{self.strategy}", current_time, Strategy=self.strategy)
            return

        if self.state == "REBOOTING":
            self.reboot_timer -= 1
            if self.reboot_timer <= 0:
                self.state = "IDLE"
                self.model.logger.add_event(current_step, self.pool_name, "Reboot_System", current_time, Strategy=self.strategy)
            return

        # 3. The Process Flow (Micro-Activities)
        if self.state == "IDLE":
            self.state = "PREPARING"
            self.model.logger.add_event(current_step, self.pool_name, "Receive_Mempool_Txs", current_time, Strategy=self.strategy)
            
        elif self.state == "PREPARING":
            self.model.logger.add_event(current_step, self.pool_name, "Verify_Signatures", current_time, Strategy=self.strategy)
            self.model.logger.add_event(current_step, self.pool_name, "Assemble_Merkle_Tree", current_time + timedelta(seconds=30), Strategy=self.strategy)
            self.state = "HASHING"
            
        elif self.state == "HASHING":
            self.model.logger.add_event(current_step, self.pool_name, f"Compute_Hash_{profit_status}", current_time, Strategy=self.strategy)
            
            # 4. Finding the Block & Broadcasting
            if random.random() < self.hash_power:
                self.model.logger.add_event(current_step, self.pool_name, "Find_Proof_of_Work", current_time, Strategy=self.strategy)
                self.state = "IDLE" 
                
                # Selfish Miner Strategy (Intentional deviation)
                if self.strategy == 'selfish':
                    self.model.logger.add_event(current_step, self.pool_name, "Delay_Broadcast", current_time, Strategy=self.strategy)
                    delay = timedelta(minutes=random.randint(4, 7))
                    self.model.pending_blocks.append((self, self.pool_name, current_time + delay))
                else:
                    latency = timedelta(seconds=random.randint(1, 10))
                    self.model.logger.add_event(current_step, self.pool_name, "Broadcast_Block", current_time + latency, Strategy=self.strategy)
                    self.model.pending_blocks.append((self, self.pool_name, current_time + latency))

# --- BLOCKCHAIN MODEL ---
class BitcoinMiningModel(mesa.Model):
    def __init__(self):
        super().__init__() 
        self.logger = ProcessLogger()
        
        # CHANGED: We completely remove mesa.time and track steps manually
        self.steps = 0 
        
        self.current_time = datetime(2026, 1, 1, 0, 0, 0)
        self.btc_price = 60000 
        self.pending_blocks = []
        
        pool_configs = [
            ("Pool_A_Giant", 0.08, 500, "honest"),
            ("Pool_B_Green", 0.05, 200, "honest"),
            ("Pool_C_Opportunist", 0.05, 800, "opportunist"),
            ("Pool_D_Selfish", 0.06, 500, "selfish"),
            ("Pool_E_Small", 0.03, 500, "honest")
        ]
        
        for name, hash_pwr, cost, strat in pool_configs:
            # Mesa 3.0+ automatically adds initialized agents to self.agents
            MinerAgent(self, name, hash_pwr, cost, strat)

    def step(self):
        self.btc_price += random.uniform(-1500, 1500)
        self.current_time += timedelta(minutes=1)
        
        # CHANGED: Manually advancing our custom step counter
        self.steps += 1 
        self.agents.shuffle_do("step")
        
        # Network resolves who won the block
        if self.pending_blocks:
            ready_blocks = [b for b in self.pending_blocks if b[2] <= self.current_time]
            
            if ready_blocks:
                ready_blocks.sort(key=lambda x: x[2])
                winner, agent_case, broadcast_time = ready_blocks[0]
                
                # Log outcome for the specific agent case
                self.logger.add_event(self.steps, agent_case, "Network_Validates_Winner", broadcast_time + timedelta(seconds=10), Strategy=winner.strategy)
                winner.btc_balance += 3.125
                
                for loser, loser_case, loser_time in self.pending_blocks:
                    if loser != winner:
                        self.logger.add_event(self.steps, loser_case, "Block_Orphaned", loser_time, Strategy=loser.strategy)
                
                self.pending_blocks = []
                for agent in self.agents:
                    if agent.state != "REBOOTING":
                        agent.state = "IDLE"

# --- RUN SIMULATION ---
model = BitcoinMiningModel()

print("Simulating 1 week of agent activity. This may take a few seconds...")
for i in range(10080): 
    model.step()

event_log_df = model.logger.to_dataframe()
print(f"\nSimulation complete! Generated {len(event_log_df)} events.")
print(event_log_df.head(10))

event_log_df.to_csv("bitcoin_agent_event_log.csv", index=False)
print("\nFile saved as 'bitcoin_agent_event_log.csv'. It is ready for PM4Py/Disco!")